# 02 — FAISS Index + Baselines (B1 / B2 / B3)

Builds MiniLM embeddings + FAISS index, then smoke-tests LLM-only, Standard RAG, and Fixed Temporal RAG.

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
if not (ROOT / 'src').exists():
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)

# Colab tip:
# !pip install -q sentence-transformers faiss-cpu pandas torch transformers

In [ ]:
from src.embed_index import build_index_from_corpus

index = build_index_from_corpus(
    ROOT / 'data' / 'corpus.jsonl',
    ROOT / 'artifacts',
)
print('Indexed', len(index.docs), 'docs')

In [ ]:
from src.generate import Generator
from src.pipelines import FixedTemporalRAGPipeline, LLMOnlyPipeline, StandardRAGPipeline
from src.data_schema import load_queries

gen = Generator(backend='extractive')  # swap to 'hf' or 'api' on Colab if desired
b1 = LLMOnlyPipeline(generator=gen)
b2 = StandardRAGPipeline(index=index, generator=gen, top_k=5)
b3 = FixedTemporalRAGPipeline(index=index, generator=gen, top_k=5, alpha=0.6)

smoke = [
    'Who is the current Prime Minister of India?',
    'Who was the President of the United States in 2018?',
    'Define photosynthesis.',
    'What is 2 + 2?',
]
for q in smoke:
    print('\n===', q)
    for pipe in (b1, b2, b3):
        r = pipe.run(q)
        print(pipe.name, '->', r.answer[:120], f'(ret={r.did_retrieve}, {r.latency_s:.3f}s)')